# Groups Exercise Companion Notebook

**6.7970/8.750 Symmetry and its Application to Machine Learning**

This notebook follows the Groups exercise section by section. Use it to **prototype your code** and **test your implementations** against the course library before submitting on the website.

Each section includes small tests you can use to check your work.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atomicarchitects/symm4ml-colabs/blob/main/groups_companion.ipynb)

## Setup

In [1]:
%%capture
!pip install https://symm4ml.mit.edu/_static/symm4ml_s26/symm4ml/symm4ml_latest.zip

In [2]:
import itertools
import numpy as np
from IPython.display import HTML

from symm4ml import groups, groups_fast, plot, vis

### Reference data

These tables and matrices are used throughout the exercise for testing.

In [3]:
# The 2x2 rotation/reflection representation of P(3) from Dresselhaus
E = np.eye(2)
A = np.array([[-1., 0.], [0., 1.]])
B = np.array([[1./2., -np.sqrt(3.)/2.], [-np.sqrt(3.)/2., -1/2.]])
C = np.array([[1./2., np.sqrt(3.)/2.], [np.sqrt(3.)/2., -1/2.]])
D = np.array([[-1./2., np.sqrt(3.)/2.], [-np.sqrt(3.)/2., -1/2.]])
F = np.array([[-1./2., -np.sqrt(3.)/2.], [np.sqrt(3.)/2., -1/2.]])
p3_dresselhaus = np.stack([E, A, B, C, D, F], axis=0)

# Reference multiplication tables used in tests
ans_table1 = np.array([[0,1,2,3,4,5],[1,0,3,2,5,4],[2,5,0,4,3,1],[3,4,1,5,2,0],[4,3,5,1,0,2],[5,2,4,0,1,3]])
ans_table2 = np.array([[0,1,2,3],[1,0,3,2],[2,3,0,1],[3,2,1,0]])
ans_table3 = np.array([[0,1,2,3],[1,0,3,2],[2,3,1,0],[3,2,0,1]])
ans_table4 = np.array([[0,1,2,3],[1,2,3,0],[2,3,0,1],[3,0,1,2]])

---
## Section 1: From Matrices to Groups

### Context: Two representations of $P(3)$

$P(3)$ ($\cong D_3$) is one of the simplest nonabelian groups. We can represent it as $3\times 3$ permutation matrices or $2\times 2$ rotation/reflection matrices.

In [4]:
p3_perm = groups.permutation_matrices(3)
print(f"P(3) permutation matrices: {p3_perm.shape}")
HTML(plot.matrix_grid(p3_perm, labels=["E","A","B","C","D","F"], cell_size=20))

P(3) permutation matrices: (6, 3, 3)


1,0,0
0,1,0
0,0,1
1,0,0
0,0,1
0,1,0
0,1,0
1,0,0
0,0,1
0,1,0
0,0,1


In [5]:
print(f"D3 rotation/reflection matrices: {p3_dresselhaus.shape}")
HTML(plot.matrix_grid(p3_dresselhaus, labels=["E","A","B","C","D","F"], cell_size=30))

D3 rotation/reflection matrices: (6, 2, 2)


1,0
0,1
-1,0
0,1
½,-√3/2
-√3/2,-½
½,√3/2
√3/2,-½
-½,√3/2
-√3/2,-½
-½,-√3/2


### 1.1 `permutation_matrices(n)`

Write your implementation here, then test against the course version.

In [6]:
def permutation_matrices(n):
    """Generates all permutation matrices of n elements
    Input:
        n: int
    Output:
        matrices: np.array of shape [n!, n, n]
    """
    answer = []
    spots = list(itertools.permutations(range(0, n),n))
    for spot in spots:
      empty_matrix = np.zeros((n,n), dtype=np.int_)
      for i, v in enumerate(spot):
        empty_matrix[i, v] = 1
      answer.append(empty_matrix)

    return np.array(answer)

In [7]:
# Small tests from the course library
# Matrices can be returned in any order, so we sort before comparing
result_2 = permutation_matrices(2)
assert result_2.shape == (2, 2, 2), f"Expected shape (2, 2, 2), got {result_2.shape}"
np.testing.assert_allclose(
    np.unique(result_2, axis=0),
    np.unique(np.array([[[1, 0], [0, 1]], [[0, 1], [1, 0]]]), axis=0),
)

result_3 = permutation_matrices(3)
assert result_3.shape == (6, 3, 3), f"Expected shape (6, 3, 3), got {result_3.shape}"
np.testing.assert_allclose(
    np.unique(result_3, axis=0),
    np.unique(groups.permutation_matrices(3), axis=0),
)
print("permutation_matrices tests passed!")

permutation_matrices tests passed!


### 1.2 `generate_group(matrices)`

Use closure under multiplication to generate a full group from a subset of elements.

In [8]:
def generate_group(matrices, decimals=4):
    """Generate new group elements from matrices (group representations)
    Input:
        matrices: np.array of shape [n, d, d] of known elements
        decimals: int number of decimals to round to when comparing matrices
    Output:
        group: np.array of shape [m, d, d], where m is the size of the resultant group
    """
    keep_trying = True
    while keep_trying:
        matrix_adding = matrices.copy()
        for pair in itertools.product(matrices, repeat=2):
            new_matrix = np.round(np.dot(pair[0], pair[1]), decimals)
            matrix_adding = np.vstack((matrix_adding, [new_matrix]))
            print(f"len of new_matrices = {len(matrix_adding)}")
        new_matrices = np.unique(matrix_adding, axis=0)
        if len(new_matrices) == len(matrices):
            keep_trying = False
        matrices = new_matrices.copy()
    return matrices
    # your code here

In [9]:
# Small tests: generating P(3) from subsets
p3 = groups.permutation_matrices(3)
np.testing.assert_allclose(
    np.unique(generate_group(p3[:-2]), axis=0),
    np.unique(p3, axis=0),
)
print("generate_group tests passed!")

len of new_matrices = 5
len of new_matrices = 6
len of new_matrices = 7
len of new_matrices = 8
len of new_matrices = 9
len of new_matrices = 10
len of new_matrices = 11
len of new_matrices = 12
len of new_matrices = 13
len of new_matrices = 14
len of new_matrices = 15
len of new_matrices = 16
len of new_matrices = 17
len of new_matrices = 18
len of new_matrices = 19
len of new_matrices = 20
len of new_matrices = 7
len of new_matrices = 8
len of new_matrices = 9
len of new_matrices = 10
len of new_matrices = 11
len of new_matrices = 12
len of new_matrices = 13
len of new_matrices = 14
len of new_matrices = 15
len of new_matrices = 16
len of new_matrices = 17
len of new_matrices = 18
len of new_matrices = 19
len of new_matrices = 20
len of new_matrices = 21
len of new_matrices = 22
len of new_matrices = 23
len of new_matrices = 24
len of new_matrices = 25
len of new_matrices = 26
len of new_matrices = 27
len of new_matrices = 28
len of new_matrices = 29
len of new_matrices = 30
len of n

In [10]:
# Visualize: generating D3 from a mirror and rotation
generators = p3_dresselhaus[[1, 4]]  # A (mirror) and D (rotation)
p3_generated = groups.generate_group(generators)
print(f"Generated {len(p3_generated)} elements from 2 generators")
HTML(plot.matrix_grid(p3_generated, cell_size=30))

Generated 6 elements from 2 generators


-1,0
0,1
-½,-√3/2
√3/2,-½
-½,√3/2
-√3/2,-½
½,-√3/2
-√3/2,-½
½,√3/2
√3/2,-½
1,0


### 1.3 `cyclic_matrices(n)`

Generate cyclic group matrices using a single generator and `generate_group`.

In [11]:
def cyclic_matrices(n):
    """Generates all cyclic matrices of n elements
    Input:
        n: int
    Output:
        matrices: np.array of shape [n, n, n]
    """
    # first turn will be 0->1, 1->2, ..., n->0
    
    result = [np.identity(n)]
    for i in range(1, n):
        next_matrix = np.zeros([n,n])
        

    # YOUR CODE HERE
    pass

In [12]:
temp = groups.cyclic_matrices(4)
temp

array([[[0., 0., 0., 1.],
        [1., 0., 0., 0.],
        [0., 1., 0., 0.],
        [0., 0., 1., 0.]],

       [[0., 0., 1., 0.],
        [0., 0., 0., 1.],
        [1., 0., 0., 0.],
        [0., 1., 0., 0.]],

       [[0., 1., 0., 0.],
        [0., 0., 1., 0.],
        [0., 0., 0., 1.],
        [1., 0., 0., 0.]],

       [[1., 0., 0., 0.],
        [0., 1., 0., 0.],
        [0., 0., 1., 0.],
        [0., 0., 0., 1.]]])

In [13]:
# Quick check: C_1, C_2, C_3 should have 1, 2, 3 elements
for n in [1, 2, 3]:
    result = cyclic_matrices(n)
    assert result.shape == (n, n, n), f"cyclic_matrices({n}) shape should be ({n},{n},{n}), got {result.shape}"
print("cyclic_matrices tests passed!")

AttributeError: 'NoneType' object has no attribute 'shape'

### 1.4 `make_multiplication_table(matrices)`

Build the Cayley table: entry at row $g$, column $h$ gives the index of $g \circ h$.

In [14]:
def make_multiplication_table(matrices: np.ndarray, *, tol: float=1e-08) -> np.ndarray:
    """Makes multiplication table for group.
    Input:
        matrices: np.array of shape [n, d, d], n matrices of dimension d that form a group under matrix multiplication.
        tol: float numberical tolerance
    Output:
        Group multiplication table.
        np.array of shape [n, n] where entries correspond to indices of first dim of matrices.
    """
    n = len(matrices)
    result = np.zeros((n,n), dtype=np.int_)
    for i in range(0, n):
        result[i, 0] = i
        for j in range(1, n):
            # find matrix given by matrices[i] * matrices[j]
            result_mat = np.round(np.dot(matrices[i], matrices[j]), 8)
            # find that index in matrices
            loc = np.all(matrices == result_mat, axis=(1,2)).tolist().index(True)
            # set
            result[i, j] = loc
    return result

In [15]:
# Compare your table with the course version
table_yours = make_multiplication_table(groups.permutation_matrices(3))
table_course = groups.make_multiplication_table(groups.permutation_matrices(3))
np.testing.assert_array_equal(table_yours, table_course)
print("make_multiplication_table tests passed!")

make_multiplication_table tests passed!


In [16]:
# Visualize: the two P(3) tables look different because elements are ordered differently
table_perm = groups.make_multiplication_table(p3_perm)
table_2d = groups.make_multiplication_table(p3_dresselhaus)

HTML(plot.compare_tables(
    table_2d, table_perm,
    labels1=["E","A","B","C","D","F"],
    labels2=[str(i) for i in range(6)],
))

---
## Section 2: Group Definition

A group requires: closure, a unique identity, inverses for all elements, and associativity.

### 2.1 `identity(table)`

Find the unique identity element, or raise `ValueError("No or multiple identities")`.

In [17]:
def identity(table: np.ndarray) -> int:
    """Returns the index of the identity element.
    Input:
        table: np.array of shape [n, n] where the entry at [i, j] is the index of the product of the ith and jth elements in the group.
    Output:
        Index of identity element.
    Raises:
        ValueError("No or multiple identities") if there is no or multiple identities.
    """
    id_found = 0
    id = -1
    for i in range(0, len(table)):
        row = table[i]
        if row.tolist() == list(range(0, len(table))):
            id_found = id_found + 1
            id = i
    if id_found != 1:
        raise ValueError("No or multiple identities")
    return id

In [18]:
# Tests from the course library
assert identity(ans_table1) == 0
assert identity(ans_table2) == 0
assert identity(np.array([[1, 2, 0], [2, 0, 1], [0, 1, 2]])) == 2

# Should raise ValueError for table with no identity
try:
    identity(np.array([[0, 1, 2], [0, 1, 2], [2, 0, 1]]))
    assert False, "Should have raised ValueError"
except ValueError as e:
    assert "No or multiple identities" in str(e)

print("identity tests passed!")

identity tests passed!


### 2.2 `inverses(table)`

Return array where entry $i$ is the index of the inverse of element $i$. Raise `ValueError("Every element does not have one inverse")` if not all elements have a unique inverse.

In [19]:
def inverses(table: np.ndarray) -> np.ndarray:
    """Returns the indices of the inverses of each element.
    Input:
        table: np.array of shape [n, n] where the entry at [i, j] is the index of the product of the ith and jth elements in the group.
    Output:
        np.array of shape [n] where the ith entry is the index of the inverse of the ith element.
    Raises:
        ValueError("Every element does not have one inverse") if there is no or multiple inverses.
    """
    id = identity(table)
    n = len(table)
    answer = np.full(n, -1)
    for i in range(0, n):
        for j in range(0, n):
            if table[i, j] == id:
                if answer[i] != -1 or table[i, j] != table[j, i]:
                    raise ValueError(f"Every element does not have one inverse")
                else:
                    answer[i] = j
            elif table[i, j] == id:
                raise ValueError(f"Every element does not have one inverse")
    if np.any(answer == -1):
        raise ValueError("Every element does not have one inverse")
    return answer

In [20]:
# Tests
t = np.array([[0,1,2,3],[1,0,3,2],[2,3,0,1],[3,2,1,0]])
inv = inverses(t)
assert np.all(t[inv, np.arange(4)] == identity(t))

# Should raise ValueError
try:
    inverses(np.array([[2, 0, 0], [2, 2, 1], [0, 1, 2]]))
    assert False, "Should have raised ValueError"
except ValueError as e:
    assert "inverse" in str(e).lower()

print("inverses tests passed!")

inverses tests passed!


### 2.3 `is_closed(table)`

In [21]:
def is_closed(table: np.ndarray) -> bool:
    """Tests whether the multiplication table is closed.
    Input:
        table: np.array of shape [n, n] where the entry at [i, j] is the index of the product of the ith and jth elements in the group.
    Output:
        True if the table represents a closed binary operation, False otherwise.
    """
    n = len(table)
    for i in range(0, n):
        for j in range(0, n):
            if table[i, j] >= n:
                return False
    return True

In [22]:
assert is_closed(np.array([[0, 0], [0, 0]])) == True
assert is_closed(np.array([[1, 2], [3, 4]])) == False
print("is_closed tests passed!")

is_closed tests passed!


### 2.4 `is_associative(table)`

In [23]:
def is_associative(table: np.ndarray) -> bool:
    """Tests whether the multiplication table is associative.
    Input:
        table: np.array of shape [n, n] where the entry at [i, j] is the index of the product of the ith and jth elements in the group.
    Output:
        True if the table represents an associative binary operation, False otherwise.
    """
    associative = True
    n = len(table)
    for i in range(0, n):
        for j in range(0, n):
            for k in range(0, n):
                if i != j or i != k or j != k:
                    if table[table[i,j], k] != table[i, table[j, k]]:
                        associative = False
                        return associative
    return associative

In [24]:
assert is_associative(np.array([[0, 0, 0], [0, 0, 0], [0, 0, 0]])) == True
assert is_associative(np.array([[0, 1, 0], [0, 0, 0], [0, 0, 0]])) == False
print("is_associative tests passed!")

is_associative tests passed!


### 2.5 `test_group(table)`

Combine all four checks. Should raise specific `ValueError` messages for each failure.

In [25]:
def test_group(table: np.ndarray):
    """Tests whether the multiplication table is valid.
    Input:
        table: np.array of shape [n, n] where the entry at [i, j] is the index of the product of the ith and jth elements in the group.
    Raises:
        ValueError("Invalid indices") if the table contains invalid indices (is not closed).
        ValueError("No or multiple identities") if the table does not contain exactly one identity.
        ValueError("Every element does not have one inverse") if not every element has an inverse.
        ValueError("Not associative") if the table is not associative.
    """
    if not is_closed(table):
        raise ValueError("Invalid indices")
    identity(table)
    inverses(table)
    if not is_associative(table):
        raise ValueError("Not associative")

In [26]:
# Helper to capture ValueError messages
def value_error(fn, *args):
    try:
        fn(*args)
    except ValueError as e:
        return " ".join(map(str, e.args))
    return None

# Valid group tables should pass
t = np.array([[0,1,2,3],[1,0,3,2],[2,3,0,1],[3,2,1,0]])
test_group(t)  # should not raise
test_group(ans_table1)  # should not raise

# Invalid tables should raise specific errors
assert value_error(test_group, np.array([[0, 1, 3], [1, 2, 0], [2, 0, 1]])) == "Invalid indices"
assert value_error(test_group, np.array([[0, 1, 2], [1, 2, 0], [2, 0, 2]])) == "Not associative"

print("test_group tests passed!")

test_group tests passed!


In [27]:
# Explore group properties interactively
# Use the tabs: Elements, Rearrangement, Inverses, Subgroups, Conjugacy
HTML(plot.multiplication_table(table_2d, labels=["E","A","B","C","D","F"]))

∘,E,A,B,C,D,F
E,E,A,B,C,D,F
A,A,E,D,F,B,C
B,B,F,E,D,C,A
C,C,D,F,E,A,B
D,D,C,A,B,F,E
F,F,B,C,A,E,D


---
## Section 3: Subgroups

By Lagrange's theorem, the order of a subgroup divides the order of the group. Use `itertools.combinations` to search over subsets of the right sizes.

### 3.1 `subgroups(table)`

The course provides `groups.factors(n)`. Use it to find candidate subgroup sizes.

In [28]:
import math
def factors_mine(n):
    divisors = set()
    
    # Loop runs up to square root of n
    for i in range(1, int(math.sqrt(n)) + 1):
        if n % i == 0:
            
            # If both divisors are same (perfect square), add only once
            if n // i == i:
                divisors.add(i)
            else:
                
                # Add both divisors
                divisors.add(i)
                divisors.add(n // i)
    return divisors

In [29]:
# factors is provided for you
assert groups.factors(12) == {1, 2, 3, 4, 6, 12}
assert groups.factors(6) == {1, 2, 3, 6}
print("Factors of 6:", groups.factors(6))

Factors of 6: {1, 2, 3, 6}


In [30]:
def subgroups(table):
    """Find all subgroups of group.
    Input:
        table: np.array of shape [n, n]
    Output:
        Set of frozensets of element indices.
    """
    n = len(table)
    result = {frozenset(list(range(0, n))), frozenset({identity(table)})}
    fs = groups.factors(n)
    fs.remove(n) # already handled above
    fs.remove(1) # can only be the identity, also handled above
    for factor in fs:
        for subset in itertools.combinations(list(range(0,n)), factor):
            table_m = table.copy()
            for i in range(n-1, -1, -1):
                if i in subset:
                    table_m[table_m == i] = subset.index(i)
                else:
                    table_m = np.delete(table_m, i, axis=0)
                    table_m = np.delete(table_m, i, axis=1)
            
            try:
                test_group(table_m)
            except ValueError:
                continue
            result.add(frozenset(subset))
    return result

In [31]:
t = np.array([[0,1,2,3],[1,0,3,2],[2,3,0,1],[3,2,1,0]])
s = subgroups(t)
s

{frozenset({2, 3}),
 frozenset({0}),
 frozenset({0, 3}),
 frozenset({0, 1}),
 frozenset({0, 2}),
 frozenset({0, 1, 2, 3})}

{frozenset({0, 1}), frozenset({2, 3}), frozenset({0}), frozenset({0, 1, 2, 3})}

In [32]:
t = np.array([[0,1,2,3],[1,0,3,2],[2,3,0,1],[3,2,1,0]])
subgroups(t)
assert subgroups(t) == {
    frozenset({0}),
    frozenset({0, 1}),
    frozenset({0, 2}),
    frozenset({0, 3}),
    frozenset({0, 1, 2, 3}),
}
print("subgroups tests passed!")

AssertionError: 

### 3.2 Questions: $C_3$ and $\mathbb{Z}_2$ in $P(3)$

Use the course implementations to find which indices of `permutation_matrices(3)` form $C_3$ and $\mathbb{Z}_2$.

Utility functions `groups.remap_to_minimal` and `groups.subgroup_table_from_group_table` may be helpful.

In [33]:
p_3 = table_perm
p_3

array([[0, 1, 2, 3, 4, 5],
       [1, 0, 3, 2, 5, 4],
       [2, 4, 0, 5, 1, 3],
       [3, 5, 1, 4, 0, 2],
       [4, 2, 5, 0, 3, 1],
       [5, 3, 4, 1, 2, 0]], dtype=int32)

In [34]:
p_3_subgroups = subgroups(p_3)
p_3_subgroups

{frozenset({0}),
 frozenset({0, 3, 4}),
 frozenset({0, 1}),
 frozenset({0, 2}),
 frozenset({0, 5}),
 frozenset({0, 1, 2, 3, 4, 5})}

In [35]:
p3_subgroups = groups.subgroups(table_perm)
print("Subgroups of P(3):")
for sg in sorted(p3_subgroups, key=lambda s: (len(s), min(s))):
    print(f"  {sorted(sg)}  (order {len(sg)})")

# Visualize subgroup structure
HTML(plot.structure_explorer(table_perm, labels=[str(i) for i in range(6)]))

Subgroups of P(3):
  [0]  (order 1)
  [0, 1]  (order 2)
  [0, 2]  (order 2)
  [0, 5]  (order 2)
  [0, 3, 4]  (order 3)
  [0, 1, 2, 3, 4, 5]  (order 6)


In [36]:
# YOUR ANSWERS:
# c3_in_p3 = {(0, 3, 4)}  # sorted tuple of indices forming C_3
# z2_in_p3 = {(0, 1), (0, 2), (0, 5)}  # sorted tuples of indices forming Z_2 copies

---
## Section 4: Cosets

Left cosets: $gH = \{gh : h \in H\}$. Right cosets: $Hg = \{hg : h \in H\}$.

### 4.1 `right_coset(table, subgroup_indices)`

In [37]:
def right_coset(table, subgroup_indices):
    """Returns the right coset of the ith element.
    Input:
        table: np.array of shape [n, n] where the entry at [i, j] is the index of the product of the ith and jth elements in the group.
        subgroup_indices: Indices of elements in the subgroup.
    Output:
        Set of right cosets for each element in the group. Each coset is represented as a frozenset of indices.
    Example:
        right_coset(np.array([[0, 1], [1, 0]]), {0}) == {frozenset({1}), frozenset({0})}
    """
    n = len(table)
    result = {frozenset(subgroup_indices)}
    for i in range(0, n):
        new_s = []
        for s in subgroup_indices:
            new_s.append(table[s][i])
        if frozenset(new_s) not in result:
            result.add(frozenset(new_s))
    return result

In [38]:
t = np.array([[0,1,2,3],[1,0,3,2],[2,3,0,1],[3,2,1,0]])
assert right_coset(t, {0, 1}) == {frozenset({0, 1}), frozenset({2, 3})}
assert right_coset(t, {0, 2}) == {frozenset({0, 2}), frozenset({1, 3})}
print("right_coset tests passed!")

right_coset tests passed!


### 4.2 `left_coset(table, subgroup_indices)`

In [39]:
def left_coset(table, subgroup_indices):
    """Returns the left coset of the ith element.
    Input:
        table: np.array of shape [n, n] where the entry at [i, j] is the index of the product of the ith and jth elements in the group.
        subgroup_indices: Indices of elements in the subgroup.

    Output:
        Set of left cosets for each element in the group. Each coset is represented as a set of indices.
    """
    n = len(table)
    result = {frozenset(subgroup_indices)}
    for i in range(0, n):
        new_s = []
        for s in subgroup_indices:
            new_s.append(table[i][s])
        if frozenset(new_s) not in result:
            result.add(frozenset(new_s))
    return result

In [40]:
t = np.array([[0,1,2,3],[1,0,3,2],[2,3,0,1],[3,2,1,0]])
assert left_coset(t, {0, 1}) == {frozenset({0, 1}), frozenset({2, 3})}
print("left_coset tests passed!")

left_coset tests passed!


In [41]:
# Compare left and right cosets of P(3)
# For a non-normal subgroup {E, A}, left != right cosets
print("Left cosets of {E, A}:", groups.left_coset(table_2d, {0, 1}))
print("Right cosets of {E, A}:", groups.right_coset(table_2d, {0, 1}))
print()
# For the normal subgroup C_3 = {E, D, F}, they match
print("Left cosets of {E, D, F}:", groups.left_coset(table_2d, {0, 4, 5}))
print("Right cosets of {E, D, F}:", groups.right_coset(table_2d, {0, 4, 5}))

Left cosets of {E, A}: {frozenset({np.int32(3), np.int32(4)}), frozenset({np.int32(0), np.int32(1)}), frozenset({np.int32(2), np.int32(5)})}
Right cosets of {E, A}: {frozenset({np.int32(0), np.int32(1)}), frozenset({np.int32(3), np.int32(5)}), frozenset({np.int32(2), np.int32(4)})}

Left cosets of {E, D, F}: {frozenset({np.int32(0), np.int32(4), np.int32(5)}), frozenset({np.int32(1), np.int32(2), np.int32(3)})}
Right cosets of {E, D, F}: {frozenset({np.int32(0), np.int32(4), np.int32(5)}), frozenset({np.int32(1), np.int32(2), np.int32(3)})}


---
## Section 5: Conjugacy, Classes, and Factor Groups

### 5.1 `conjugacy_classes(table)`

$b$ is conjugate to $a$ if $\exists x \in G$ such that $b = xax^{-1}$.

In [42]:
def conjugacy_classes(table: np.ndarray)-> set[frozenset[int]]:
    """Returns the conjugacy classes of the group.
    Input:
        table: np.array of shape [n, n] where the entry at [i, j] is the index of the product of the ith and jth elements in the group.
    Output:
        Set of conjugacy classes. Each conjugacy class is a set of integers.
    """
    n = len(table)
    result = set()
    inv = inverses(table)
    for i in range(0, n):
        conj = []
        for j in range(0, n):
            conj.append(table[j][table[i][inv[j]]])
        result.add(frozenset(conj))
    return result

In [43]:
# D2 (abelian) — every element is its own class
t = np.array([[0,1,2,3],[1,0,3,2],[2,3,0,1],[3,2,1,0]])
assert conjugacy_classes(t) == {
    frozenset({0}), frozenset({1}), frozenset({2}), frozenset({3}),
}
print("conjugacy_classes tests passed!")

conjugacy_classes tests passed!


In [44]:
p3_p = groups.permutation_matrices(3)
p3_p

array([[[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]],

       [[1., 0., 0.],
        [0., 0., 1.],
        [0., 1., 0.]],

       [[0., 1., 0.],
        [1., 0., 0.],
        [0., 0., 1.]],

       [[0., 1., 0.],
        [0., 0., 1.],
        [1., 0., 0.]],

       [[0., 0., 1.],
        [1., 0., 0.],
        [0., 1., 0.]],

       [[0., 0., 1.],
        [0., 1., 0.],
        [1., 0., 0.]]])

In [45]:
# P(3) conjugacy classes: {E}, {D, F} (rotations), {A, B, C} (mirrors)
conj = groups.conjugacy_classes(table_2d)
labels = ["E","A","B","C","D","F"]
for c in sorted(conj, key=lambda s: (len(s), min(s))):
    print("{"+", ".join(labels[i] for i in sorted(c))+"}")

{E}
{D, F}
{A, B, C}


### 5.2 `selfconjugate_subgroups(table)`

A subgroup $H$ is self-conjugate (normal) if $gHg^{-1} = H$ for all $g \in G$.

In [46]:
t = np.array([[0,1,2,3],[1,0,3,2],[2,3,0,1],[3,2,1,0]])
t

array([[0, 1, 2, 3],
       [1, 0, 3, 2],
       [2, 3, 0, 1],
       [3, 2, 1, 0]])

In [47]:
subgroups(t)

{frozenset({2, 3}),
 frozenset({0}),
 frozenset({0, 3}),
 frozenset({0, 1}),
 frozenset({0, 2}),
 frozenset({0, 1, 2, 3})}

In [48]:
conjugacy_classes(t)

{frozenset({np.int64(3)}),
 frozenset({np.int64(2)}),
 frozenset({np.int64(1)}),
 frozenset({np.int64(0)})}

In [49]:
def selfconjugate_subgroups(table: np.ndarray) -> set[frozenset[int]]:
    """Returns the set of self-conjugate (normal) subgroups."""
    sgs = subgroups(table)
    inv = inverses(table)
    result = set()
    for sg in sgs:
        is_self_conjugate = True
        for h in sg:
            for g in range(0, len(table)):
                if table[g][table[h][inv[g]]] not in sg:
                    is_self_conjugate = False
                    break
        if is_self_conjugate:
            result.add(frozenset(sg))

    return result

In [50]:
# D2 is abelian so all subgroups are normal
t = np.array([[0,1,2,3],[1,0,3,2],[2,3,0,1],[3,2,1,0]])
print(selfconjugate_subgroups(t))
assert selfconjugate_subgroups(t) == {
    frozenset({0}),
    frozenset({0, 1}),
    frozenset({0, 2}),
    frozenset({0, 3}),
    frozenset({0, 1, 2, 3}),
}
print("selfconjugate_subgroups tests passed!")

{frozenset({2, 3}), frozenset({0, 3}), frozenset({0, 1}), frozenset({0, 2}), frozenset({0, 1, 2, 3}), frozenset({0})}


AssertionError: 

In [92]:
print(groups.permutation_matrices(3))

[[[1. 0. 0.]
  [0. 1. 0.]
  [0. 0. 1.]]

 [[1. 0. 0.]
  [0. 0. 1.]
  [0. 1. 0.]]

 [[0. 1. 0.]
  [1. 0. 0.]
  [0. 0. 1.]]

 [[0. 1. 0.]
  [0. 0. 1.]
  [1. 0. 0.]]

 [[0. 0. 1.]
  [1. 0. 0.]
  [0. 1. 0.]]

 [[0. 0. 1.]
  [0. 1. 0.]
  [1. 0. 0.]]]


In [93]:
groups.selfconjugate_subgroups(ans_table1)

{frozenset({0}), frozenset({0, 3, 5}), frozenset({0, 1, 2, 3, 4, 5})}

In [95]:
ans_table1

array([[0, 1, 2, 3, 4, 5],
       [1, 0, 3, 2, 5, 4],
       [2, 5, 0, 4, 3, 1],
       [3, 4, 1, 5, 2, 0],
       [4, 3, 5, 1, 0, 2],
       [5, 2, 4, 0, 1, 3]])

### 5.3 `factor_group(table, selfconj_sub)`

The factor group $G/H$ treats each coset of $H$ as a single element.

In [51]:
def factor_group(table, selfconj_sub):
    """Returns the factor group of the group.
    Input:
        table: np.array of shape [n, n] where entries correspond to indices of group elements.
        selfconj_sub: set of indices for self-conjugate subgroup.
    Output:
        Multiplication table of factor group of order n2 as sets of  elements of the group
        np.array sets of ints of shape [n2, n2]
        Multiplication table of factor group in terms of indices of right cosests
        np.array of shape [n2, n2] where entries correspond to indices of first dim of matrices.
    """
    coset_table = list(right_coset(table, selfconj_sub))
    coset_table2 = left_coset(table, selfconj_sub)
    print(f"coset table = {coset_table}")
    #int_table = np.empty([len(coset_table), len(coset_table)])
    #just_other_cosets = coset_table.copy()
    #just_other_cosets.remove(selfconj_sub)
    #right_order = [selfconj_sub] + list(just_other_cosets)
    vals_order = list(itertools.chain.from_iterable(coset_table))

    new_table = table.copy()[vals_order][:,vals_order]
    #new_table = groups.permute_mul_table(table, vals_order)

    int_table = []
    for i in range(0, len(coset_table)):
        new_row = []
        for j in range(0, len(coset_table)):
            c_1 = list(coset_table[i])
            c_2 = list(coset_table[j])

            val = table[c_1[0], c_2[0]]
            for k in coset_table:
                if val in k:
                    print(f"c_1 is {c_1}, c_2 is {c_2}, val is [{c_1[0]},{c_2[0]}] = {val}, k is {k}")
                    #int_table[i][j] = coset_table.index(k)
                    new_row.append(coset_table.index(k))
                    break
        int_table.append(new_row)

    return ([list(coset_table), list(coset_table2)], int_table)

In [52]:
ans = factor_group(ans_table1, frozenset({0, 3, 5}))
print(ans)

coset table = [frozenset({np.int64(1), np.int64(2), np.int64(4)}), frozenset({0, 3, 5})]
c_1 is [np.int64(1), np.int64(2), np.int64(4)], c_2 is [np.int64(1), np.int64(2), np.int64(4)], val is [1,1] = 0, k is frozenset({0, 3, 5})
c_1 is [np.int64(1), np.int64(2), np.int64(4)], c_2 is [0, 3, 5], val is [1,0] = 1, k is frozenset({np.int64(1), np.int64(2), np.int64(4)})
c_1 is [0, 3, 5], c_2 is [np.int64(1), np.int64(2), np.int64(4)], val is [0,1] = 1, k is frozenset({np.int64(1), np.int64(2), np.int64(4)})
c_1 is [0, 3, 5], c_2 is [0, 3, 5], val is [0,0] = 0, k is frozenset({0, 3, 5})
([[frozenset({np.int64(1), np.int64(2), np.int64(4)}), frozenset({0, 3, 5})], [frozenset({np.int64(1), np.int64(2), np.int64(4)}), frozenset({0, 3, 5})]], [[1, 0], [0, 1]])


In [53]:
ans = factor_group(ans_table1, frozenset({0, 3, 5}))
print(ans)
np.testing.assert_array_equal(ans[1], [[1,0],[0,1]])

ans = factor_group(ans_table2, frozenset({0,1}))[1]
np.testing.assert_array_equal(ans, [[0,1],[1,0]])

coset table = [frozenset({np.int64(1), np.int64(2), np.int64(4)}), frozenset({0, 3, 5})]
c_1 is [np.int64(1), np.int64(2), np.int64(4)], c_2 is [np.int64(1), np.int64(2), np.int64(4)], val is [1,1] = 0, k is frozenset({0, 3, 5})
c_1 is [np.int64(1), np.int64(2), np.int64(4)], c_2 is [0, 3, 5], val is [1,0] = 1, k is frozenset({np.int64(1), np.int64(2), np.int64(4)})
c_1 is [0, 3, 5], c_2 is [np.int64(1), np.int64(2), np.int64(4)], val is [0,1] = 1, k is frozenset({np.int64(1), np.int64(2), np.int64(4)})
c_1 is [0, 3, 5], c_2 is [0, 3, 5], val is [0,0] = 0, k is frozenset({0, 3, 5})
([[frozenset({np.int64(1), np.int64(2), np.int64(4)}), frozenset({0, 3, 5})], [frozenset({np.int64(1), np.int64(2), np.int64(4)}), frozenset({0, 3, 5})]], [[1, 0], [0, 1]])
coset table = [frozenset({0, 1}), frozenset({np.int64(2), np.int64(3)})]
c_1 is [0, 1], c_2 is [0, 1], val is [0,0] = 0, k is frozenset({0, 1})
c_1 is [0, 1], c_2 is [np.int64(2), np.int64(3)], val is [0,2] = 2, k is frozenset({np.int64(2

In [54]:
ans = factor_group(ans_table1, frozenset({0, 3, 5}))
print(f"my ans is \n {ans[0]} \n ---- \n {ans[1]}")

coset table = [frozenset({np.int64(1), np.int64(2), np.int64(4)}), frozenset({0, 3, 5})]
c_1 is [np.int64(1), np.int64(2), np.int64(4)], c_2 is [np.int64(1), np.int64(2), np.int64(4)], val is [1,1] = 0, k is frozenset({0, 3, 5})
c_1 is [np.int64(1), np.int64(2), np.int64(4)], c_2 is [0, 3, 5], val is [1,0] = 1, k is frozenset({np.int64(1), np.int64(2), np.int64(4)})
c_1 is [0, 3, 5], c_2 is [np.int64(1), np.int64(2), np.int64(4)], val is [0,1] = 1, k is frozenset({np.int64(1), np.int64(2), np.int64(4)})
c_1 is [0, 3, 5], c_2 is [0, 3, 5], val is [0,0] = 0, k is frozenset({0, 3, 5})
my ans is 
 [[frozenset({np.int64(1), np.int64(2), np.int64(4)}), frozenset({0, 3, 5})], [frozenset({np.int64(1), np.int64(2), np.int64(4)}), frozenset({0, 3, 5})]] 
 ---- 
 [[1, 0], [0, 1]]


In [55]:
groups_ex = groups.factor_group(ans_table1, frozenset({0,3,5}))
print(f"for table \n{ans_table1} \n ---- \ntrue ans is \n {groups_ex[0]} \n ---- \n {groups_ex[1]}")

for table 
[[0 1 2 3 4 5]
 [1 0 3 2 5 4]
 [2 5 0 4 3 1]
 [3 4 1 5 2 0]
 [4 3 5 1 0 2]
 [5 2 4 0 1 3]] 
 ---- 
true ans is 
 [[{np.int64(0), np.int64(3), np.int64(5)}
  {np.int64(1), np.int64(2), np.int64(4)}]
 [{np.int64(1), np.int64(2), np.int64(4)}
  {np.int64(0), np.int64(3), np.int64(5)}]] 
 ---- 
 [[1 0]
 [0 1]]


In [56]:
ans_table2

array([[0, 1, 2, 3],
       [1, 0, 3, 2],
       [2, 3, 0, 1],
       [3, 2, 1, 0]])

In [57]:
groups_ex_2 = groups.factor_group(ans_table2, frozenset({0,1}))
print(f"true ans is \n {groups_ex_2[0]} \n ---- \n {groups_ex_2[1]}")

true ans is 
 [[{np.int64(0), np.int64(1)} {np.int64(2), np.int64(3)}]
 [{np.int64(2), np.int64(3)} {np.int64(0), np.int64(1)}]] 
 ---- 
 [[0 1]
 [1 0]]


In [58]:
# Compare your factor group with the course version
_, ft_yours = factor_group(ans_table1, frozenset({0, 3, 5}))
_, ft_course = groups.factor_group(ans_table1, frozenset({0, 3, 5}))

# The tables should be isomorphic (possibly different labeling)
print("Your factor group table:")
print(ft_yours)
print("Course factor group table:")
print(ft_course)

coset table = [frozenset({np.int64(1), np.int64(2), np.int64(4)}), frozenset({0, 3, 5})]
c_1 is [np.int64(1), np.int64(2), np.int64(4)], c_2 is [np.int64(1), np.int64(2), np.int64(4)], val is [1,1] = 0, k is frozenset({0, 3, 5})
c_1 is [np.int64(1), np.int64(2), np.int64(4)], c_2 is [0, 3, 5], val is [1,0] = 1, k is frozenset({np.int64(1), np.int64(2), np.int64(4)})
c_1 is [0, 3, 5], c_2 is [np.int64(1), np.int64(2), np.int64(4)], val is [0,1] = 1, k is frozenset({np.int64(1), np.int64(2), np.int64(4)})
c_1 is [0, 3, 5], c_2 is [0, 3, 5], val is [0,0] = 0, k is frozenset({0, 3, 5})
Your factor group table:
[[1, 0], [0, 1]]
Course factor group table:
[[1 0]
 [0 1]]


In [59]:
sconj_g = groups.subgroups(table_2d)
sconj_g

{frozenset({0}),
 frozenset({0, 3}),
 frozenset({0, 4, 5}),
 frozenset({0, 1}),
 frozenset({0, 2}),
 frozenset({0, 1, 2, 3, 4, 5})}

In [60]:
tem = groups.selfconjugate_subgroups(table_2d)
tem

{frozenset({0}), frozenset({0, 4, 5}), frozenset({0, 1, 2, 3, 4, 5})}

In [61]:
list(tem)[2]

frozenset({0})

In [62]:
print(groups.factor_group(table_2d, list(tem)[1])[1])

[[0]]


In [63]:
# P(3) / C_3 ≅ Z_2
coset_labels, factor_table = groups.factor_group(table_2d, frozenset({0, 4, 5}))
print("Factor group P(3)/C_3:")
print(factor_table)
print("This is Z_2!")
print(coset_labels)

Factor group P(3)/C_3:
[[0 1]
 [1 0]]
This is Z_2!
[[{np.int32(0), np.int32(4), np.int32(5)}
  {np.int32(1), np.int32(2), np.int32(3)}]
 [{np.int32(1), np.int32(2), np.int32(3)}
  {np.int32(0), np.int32(4), np.int32(5)}]]


In [64]:
cs_lab_2, fact_t_2 = groups.factor_group(table_2d, frozenset({0}))
print("Factor group P(3)/?:")
print(fact_t_2)
print("This is ?!")
print(cs_lab_2)

Factor group P(3)/?:
[[5 4 3 2 1 0]
 [3 5 4 0 2 1]
 [4 3 5 1 0 2]
 [1 2 0 4 5 3]
 [2 0 1 5 3 4]
 [0 1 2 3 4 5]]
This is ?!
[[{np.int32(0)} {np.int32(4)} {np.int32(5)} {np.int32(1)} {np.int32(3)}
  {np.int32(2)}]
 [{np.int32(5)} {np.int32(0)} {np.int32(4)} {np.int32(2)} {np.int32(1)}
  {np.int32(3)}]
 [{np.int32(4)} {np.int32(5)} {np.int32(0)} {np.int32(3)} {np.int32(2)}
  {np.int32(1)}]
 [{np.int32(3)} {np.int32(1)} {np.int32(2)} {np.int32(4)} {np.int32(0)}
  {np.int32(5)}]
 [{np.int32(1)} {np.int32(2)} {np.int32(3)} {np.int32(0)} {np.int32(5)}
  {np.int32(4)}]
 [{np.int32(2)} {np.int32(3)} {np.int32(1)} {np.int32(5)} {np.int32(4)}
  {np.int32(0)}]]


In [65]:
cs_lab_3, fact_t_3 = groups.factor_group(table_2d, frozenset({0,1,2,3,4,5}))
print("Factor group P(3)/?:")
print(fact_t_3)
print("This is ?!")
print(cs_lab_3)

Factor group P(3)/?:
[[0]]
This is ?!
[[{np.int32(0), np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5)}]]


In [66]:
# Visualize cosets and factor groups
HTML(plot.structure_explorer(table_2d, labels=["E","A","B","C","D","F"]))

---
## Section 6: Comparing Tables

### 6.1 `isomorphisms(table_src, table_dst)`

Find all relabelings $h$ such that $h(g_1 \cdot g_2) = h(g_1) \cdot h(g_2)$.

You may find `groups.permute_mul_table` helpful for testing.

In [67]:
print(groups.permute_mul_table.__doc__)

Multiplication table of the same group with a different ordering.
    Tip: If your solution does not work, try with the inverse permutation.
    Input:
        table: np.array of shape [n, n]
        perm: np.array of shape [n]
    Output:
        permuted multiplication table.
        np.array of shape [n, n]
    


In [ ]:
def isomorphisms(table_src: np.array, table_dst: np.array)-> set[tuple[int]]:
    """Finds all isomorphisms between two multiplication tables.
    Returns a set of tuples h of length n.
    Input:
        table_src: np.array of shape [n, n] where the entry at [i, j] is the index of the product of the ith and jth elements in the source group.
        table_dst: np.array of shape [n, n] where the entry at [i, j] is the index of the product of the ith and jth elements in the destination group.
    Output:
        A set of isomorphisms encoded as tuples ``h`` of length ``n``.
        Each element ``h[i]`` is the index of the image of the ith element in the source group.
    """
    # YOUR CODE HERE
    result = set()
    n = len(table_src)
    for subset in itertools.permutations(list(range(0,n)), n):
        print(subset)
        t = groups.permute_mul_table(table_dst, np.array(subset))
        is_iso = True
        if np.all(subset == (0, 2, 3, 1)):
            print("stop here")
        for i in range(0,n):
            for j in range(0,n):
                if t[i][j] != table_src[i][j]:
                    is_iso = False
                    break
        print(t)
        print("_________")
        if is_iso:
            result.add(subset)
    return result
    
    # result is just a list of isomorphisms, where each isomorphism has length n


In [89]:
isomorphisms(ans_table3, ans_table4)

(0, 1, 2, 3)
[[0 1 2 3]
 [1 2 3 0]
 [2 3 0 1]
 [3 0 1 2]]
_________
(0, 1, 3, 2)
[[0 1 2 3]
 [1 3 0 2]
 [2 0 3 1]
 [3 2 1 0]]
_________
(0, 2, 1, 3)
[[0 1 2 3]
 [1 0 3 2]
 [2 3 1 0]
 [3 2 0 1]]
_________
(0, 2, 3, 1)
stop here
[[0 1 2 3]
 [1 0 3 2]
 [2 3 1 0]
 [3 2 0 1]]
_________
(0, 3, 1, 2)
[[0 1 2 3]
 [1 3 0 2]
 [2 0 3 1]
 [3 2 1 0]]
_________
(0, 3, 2, 1)
[[0 1 2 3]
 [1 2 3 0]
 [2 3 0 1]
 [3 0 1 2]]
_________
(1, 0, 2, 3)
[[2 0 3 1]
 [0 1 2 3]
 [3 2 1 0]
 [1 3 0 2]]
_________
(1, 0, 3, 2)
[[3 0 1 2]
 [0 1 2 3]
 [1 2 3 0]
 [2 3 0 1]]
_________
(1, 2, 0, 3)
[[1 3 0 2]
 [3 2 1 0]
 [0 1 2 3]
 [2 0 3 1]]
_________
(1, 2, 3, 0)
[[1 2 3 0]
 [2 3 0 1]
 [3 0 1 2]
 [0 1 2 3]]
_________
(1, 3, 0, 2)
[[3 2 0 1]
 [2 3 1 0]
 [0 1 2 3]
 [1 0 3 2]]
_________
(1, 3, 2, 0)
[[2 3 1 0]
 [3 2 0 1]
 [1 0 3 2]
 [0 1 2 3]]
_________
(2, 0, 1, 3)
[[1 0 3 2]
 [0 1 2 3]
 [3 2 0 1]
 [2 3 1 0]]
_________
(2, 0, 3, 1)
[[1 0 3 2]
 [0 1 2 3]
 [3 2 0 1]
 [2 3 1 0]]
_________
(2, 1, 0, 3)
[[2 3 0 1]
 [3 0 1 2]
 [0

{(0, 2, 1, 3), (0, 2, 3, 1)}

In [90]:
# Z_2 has only the identity isomorphism to itself
assert isomorphisms(np.array([[0, 1], [1, 0]]), np.array([[0, 1], [1, 0]])) == {(0, 1)}

# C_4 and D_2 are NOT isomorphic (same order, different structure)
assert isomorphisms(ans_table3, ans_table4) == set()

# D_2 to itself has multiple isomorphisms
assert len(isomorphisms(ans_table2, ans_table2)) > 0

print("isomorphisms tests passed!")

(0, 1)
[[0 1]
 [1 0]]
_________
(1, 0)
[[1 0]
 [0 1]]
_________
(0, 1, 2, 3)
[[0 1 2 3]
 [1 2 3 0]
 [2 3 0 1]
 [3 0 1 2]]
_________
(0, 1, 3, 2)
[[0 1 2 3]
 [1 3 0 2]
 [2 0 3 1]
 [3 2 1 0]]
_________
(0, 2, 1, 3)
[[0 1 2 3]
 [1 0 3 2]
 [2 3 1 0]
 [3 2 0 1]]
_________
(0, 2, 3, 1)
stop here
[[0 1 2 3]
 [1 0 3 2]
 [2 3 1 0]
 [3 2 0 1]]
_________
(0, 3, 1, 2)
[[0 1 2 3]
 [1 3 0 2]
 [2 0 3 1]
 [3 2 1 0]]
_________
(0, 3, 2, 1)
[[0 1 2 3]
 [1 2 3 0]
 [2 3 0 1]
 [3 0 1 2]]
_________
(1, 0, 2, 3)
[[2 0 3 1]
 [0 1 2 3]
 [3 2 1 0]
 [1 3 0 2]]
_________
(1, 0, 3, 2)
[[3 0 1 2]
 [0 1 2 3]
 [1 2 3 0]
 [2 3 0 1]]
_________
(1, 2, 0, 3)
[[1 3 0 2]
 [3 2 1 0]
 [0 1 2 3]
 [2 0 3 1]]
_________
(1, 2, 3, 0)
[[1 2 3 0]
 [2 3 0 1]
 [3 0 1 2]
 [0 1 2 3]]
_________
(1, 3, 0, 2)
[[3 2 0 1]
 [2 3 1 0]
 [0 1 2 3]
 [1 0 3 2]]
_________
(1, 3, 2, 0)
[[2 3 1 0]
 [3 2 0 1]
 [1 0 3 2]
 [0 1 2 3]]
_________
(2, 0, 1, 3)
[[1 0 3 2]
 [0 1 2 3]
 [3 2 0 1]
 [2 3 1 0]]
_________
(2, 0, 3, 1)
[[1 0 3 2]
 [0 1 2 3]
 [3 2 0

AssertionError: 

In [91]:
# The two P(3) representations are isomorphic
isos = groups.isomorphisms(table_2d, table_perm)
print(f"Found {len(isos)} isomorphisms from D3 to P(3) perm")

# Use one isomorphism to reorder and compare
reorder = list(list(isos)[0])
table_perm_reordered = groups.make_multiplication_table(p3_perm[reorder])
HTML(plot.compare_tables(
    table_2d, table_perm_reordered,
    labels1=["E","A","B","C","D","F"],
    labels2=["E","A","B","C","D","F"],
))

Found 6 isomorphisms from D3 to P(3) perm


In [101]:
isos

{(0, 1, 2, 5, 3, 4),
 (0, 1, 5, 2, 4, 3),
 (0, 2, 1, 5, 4, 3),
 (0, 2, 5, 1, 3, 4),
 (0, 5, 1, 2, 3, 4),
 (0, 5, 2, 1, 4, 3)}

### 6.2 `surjective_homomorphisms(table_src, table_dst)`

Like isomorphisms but the map need not be injective — only surjective.

In [ ]:
def surjective_homomorphisms(table_src, table_dst):
    """Finds all surjective homomorphisms from src to dst.
    Returns a set of tuples h of length n_src.
    """
    # YOUR CODE HERE
    pass

In [ ]:
assert surjective_homomorphisms(
    np.array([[0, 1], [1, 0]]), np.array([[0, 1], [1, 0]])
) == {(0, 1)}
print("surjective_homomorphisms tests passed!")

### $C_4$ vs $D_2$: same order, not isomorphic

$C_4$ has elements of order 4, $D_2$ does not.

In [ ]:
table_c4 = groups.make_multiplication_table(groups.cyclic_matrices(4))

isos_c4_d2 = groups.isomorphisms(table_c4, groups.D2_table)
print(f"C4 ≅ D2? {'Yes' if isos_c4_d2 else 'No — NOT isomorphic'}")

HTML(plot.compare_tables(
    table_c4, groups.D2_table,
    labels1=["e", "r", "r²", "r³"],
    labels2=["e", "a", "b", "c"],
))

---
## Section 7: Symmetries of Molecule $AB_4$

The $AB_4$ molecule has a central atom $A$ with four $B$ atoms at the corners of a square (not coplanar with $A$). Its symmetry group has 8 elements.

### 7.1 `AB4_group()`

Return $3 \times 3$ rotation and reflection matrices that leave the molecule invariant.

Coordinates: A = (0, 0, 1), B₁ = (1, 1, 0), B₂ = (−1, 1, 0), B₃ = (−1, −1, 0), B₄ = (1, −1, 0)

In [ ]:
a = [0,0,1]
b_1 = [1,1,0]
b_2 = [-1, 1, 0]
b_3 = [-1, -1, 0]
b_4 = [1, -1, 0]

In [ ]:
def AB4_group():
    """Return 3D rotation and reflection matrices for the symmetry of AB_4.
    Output: np.array of shape [N, 3, 3]
    """
    result = []
    rotations = groups.cyclic_matrices(2)
    rotations_3 = np.kron(rotations, identity(3))
    
    # YOUR CODE HERE
    # reflection along x axis
    # reflection along y axis
    # reflection along x-y 45 degree axis
    # reflection along x-y -45 degree axis
    # 90 degree rotation along z axis
    result.append([[1, 0, 0], [0, -1, 0], [0, 0, 1]])
    # 180 degree rotation along z axis
    result.append([[-1, 0, 0], [0, -1, 0], [0, 0, 1]])
    # 270 degree rotation along z axis
    result.append([[-1, 0, 0], [0, 1, 0], [0, 0, 1]])
    return np.array(result)

In [114]:
ab4 = AB4_group()
print(ab4)
assert ab4.shape[0] == 8, f"AB4 group should have 8 elements, got {ab4.shape[0]}"
assert ab4.shape[1:] == (3, 3), f"Matrices should be 3x3"

# Check all matrices are orthogonal
for m in ab4:
    np.testing.assert_allclose(m @ m.T, np.eye(3), atol=1e-8)

# Compare with course implementation
ab4_course = groups.AB4_group()
table_ab4 = groups.make_multiplication_table(ab4)
table_ab4_course = groups.make_multiplication_table(ab4_course)
assert len(groups.isomorphisms(table_ab4, table_ab4_course)) > 0, "Not isomorphic to course solution!"
print("AB4_group tests passed!")

[[[ 1  0  0]
  [ 0 -1  0]
  [ 0  0  1]]

 [[-1  0  0]
  [ 0 -1  0]
  [ 0  0  1]]

 [[-1  0  0]
  [ 0  1  0]
  [ 0  0  1]]]


AssertionError: AB4 group should have 8 elements, got 3

In [108]:
print(ab4_course)

[[[-1  0  0]
  [ 0 -1  0]
  [ 0  0  1]]

 [[-1  0  0]
  [ 0  1  0]
  [ 0  0  1]]

 [[ 0 -1  0]
  [-1  0  0]
  [ 0  0  1]]

 [[ 0 -1  0]
  [ 1  0  0]
  [ 0  0  1]]

 [[ 0  1  0]
  [-1  0  0]
  [ 0  0  1]]

 [[ 0  1  0]
  [ 1  0  0]
  [ 0  0  1]]

 [[ 1  0  0]
  [ 0 -1  0]
  [ 0  0  1]]

 [[ 1  0  0]
  [ 0  1  0]
  [ 0  0  1]]]


In [104]:
ab4_course = groups.AB4_group()
table_ab4 = groups.make_multiplication_table(ab4_course)
print(f"AB₄ symmetry group has {len(ab4_course)} elements")
HTML(plot.matrix_grid(ab4_course, cell_size=24))

AB₄ symmetry group has 8 elements


-1,0,0
0,-1,0
0,0,1
-1,0,0
0,1,0
0,0,1
0,-1,0
-1,0,0
0,0,1
0,-1,0
1,0,0


In [105]:
HTML(plot.multiplication_table(table_ab4))

∘,0,1,2,3,4,5,6,7
0,7,6,5,4,3,2,1,0
1,6,7,4,5,2,3,0,1
2,5,3,7,1,6,0,4,2
3,4,2,6,0,7,1,5,3
4,3,5,1,7,0,6,2,4
5,2,4,0,6,1,7,3,5
6,1,0,3,2,5,4,7,6
7,0,1,2,3,4,5,6,7


### 7.2 `AB4_sc_subs_iso_C4_vs_D2(AB4_matrices)`

Classify order-4 self-conjugate subgroups as $C_4$ or $D_2$.

You can access `groups.D2_table` and `groups.C4_table` directly. You may find `groups.remap_to_minimal` and `groups.subgroup_table_from_group_table` helpful.

In [ ]:
def AB4_sc_subs_iso_C4_vs_D2(AB4_matrices):
    """Returns (C4_sets, D2_sets) where each is a set of frozensets of indices."""
    # YOUR CODE HERE
    pass

In [ ]:
C4_sets, D2_sets = AB4_sc_subs_iso_C4_vs_D2(groups.AB4_group())
print(f"Self-conjugate subgroups isomorphic to C4: {C4_sets}")
print(f"Self-conjugate subgroups isomorphic to D2: {D2_sets}")

# Check against course
C4_course, D2_course = groups.AB4_sc_subs_iso_C4_vs_D2(groups.AB4_group())
assert C4_sets == C4_course and D2_sets == D2_course, "Does not match course solution!"
print("AB4_sc_subs_iso_C4_vs_D2 tests passed!")

In [116]:
print(groups.C4_table)

[[1 2 3 0]
 [2 3 0 1]
 [3 0 1 2]
 [0 1 2 3]]


In [117]:
print(groups.D2_table)

[[3 2 1 0]
 [2 3 0 1]
 [1 0 3 2]
 [0 1 2 3]]


In [115]:
HTML(plot.structure_explorer(table_ab4))

---
## Section 8: Playing with $P(4)$

$P(4)$ has $4! = 24$ elements. The naive `groups.subgroups` and `groups.isomorphisms` functions are too slow at this scale — use `groups_fast` instead.

**Important:** Use `groups_fast.generate_subgroups_dynamic_programming` and `groups_fast.isomorphisms_generator_backtracking`.

In [118]:
p4 = groups.permutation_matrices(4)
table_p4 = groups.make_multiplication_table(p4)
print(f"P(4) has {len(p4)} elements")

P(4) has 24 elements


### 8.1 Order of $P(4)$

In [119]:
p4_order = len(p4)
print(f"|P(4)| = {p4_order}")

|P(4)| = 24


### 8.2 Conjugacy classes of $P(4)$

Match each conjugacy class to its geometric interpretation: $E$, $C_2$, $C_3$, $\sigma_d$, $S_4$.

In [120]:
conj_p4 = groups.conjugacy_classes(table_p4)
print(f"{len(conj_p4)} conjugacy classes:")
for c in sorted(conj_p4, key=lambda s: (len(s), min(s))):
    print(f"  {sorted(c)}  (size {len(c)})")

5 conjugacy classes:
  [np.int32(0)]  (size 1)
  [np.int32(7), np.int32(16), np.int32(23)]  (size 3)
  [np.int32(1), np.int32(2), np.int32(5), np.int32(6), np.int32(14), np.int32(21)]  (size 6)
  [np.int32(9), np.int32(10), np.int32(13), np.int32(17), np.int32(18), np.int32(22)]  (size 6)
  [np.int32(3), np.int32(4), np.int32(8), np.int32(11), np.int32(12), np.int32(15), np.int32(19), np.int32(20)]  (size 8)


In [121]:
# Look at the actual matrices to identify each class geometrically
# Hint: check determinants (rotation vs improper) and traces
for c in sorted(conj_p4, key=lambda s: (len(s), min(s))):
    rep = sorted(c)[0]
    det = np.linalg.det(p4[rep])
    tr = np.trace(p4[rep])
    print(f"  Class {sorted(c)}: det={det:+.0f}, trace={tr:+.0f}")
    print(f"    Representative matrix:\n{p4[rep]}\n")

  Class [np.int32(0)]: det=+1, trace=+4
    Representative matrix:
[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]

  Class [np.int32(7), np.int32(16), np.int32(23)]: det=+1, trace=+0
    Representative matrix:
[[0. 1. 0. 0.]
 [1. 0. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 1. 0.]]

  Class [np.int32(1), np.int32(2), np.int32(5), np.int32(6), np.int32(14), np.int32(21)]: det=-1, trace=+2
    Representative matrix:
[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 1. 0.]]

  Class [np.int32(9), np.int32(10), np.int32(13), np.int32(17), np.int32(18), np.int32(22)]: det=-1, trace=+0
    Representative matrix:
[[0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]
 [1. 0. 0. 0.]]

  Class [np.int32(3), np.int32(4), np.int32(8), np.int32(11), np.int32(12), np.int32(15), np.int32(19), np.int32(20)]: det=+1, trace=+1
    Representative matrix:
[[1. 0. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]
 [0. 1. 0. 0.]]



In [122]:
HTML(plot.multiplication_table(table_p4, cell_size=22))

∘,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
0,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
1,1,0,3,2,5,4,7,6,9,8,11,10,13,12,15,14,17,16,19,18,21,20,23,22
2,2,4,0,5,1,3,8,10,6,11,7,9,14,16,12,17,13,15,20,22,18,23,19,21
3,3,5,1,4,0,2,9,11,7,10,6,8,15,17,13,16,12,14,21,23,19,22,18,20
4,4,2,5,0,3,1,10,8,11,6,9,7,16,14,17,12,15,13,22,20,23,18,21,19
5,5,3,4,1,2,0,11,9,10,7,8,6,17,15,16,13,14,12,23,21,22,19,20,18
6,6,7,12,13,18,19,0,1,14,15,20,21,2,3,8,9,22,23,4,5,10,11,16,17
7,7,6,13,12,19,18,1,0,15,14,21,20,3,2,9,8,23,22,5,4,11,10,17,16
8,8,10,14,16,20,22,2,4,12,17,18,23,0,5,6,11,19,21,1,3,7,9,13,15
9,9,11,15,17,21,23,3,5,13,16,19,22,1,4,7,10,18,20,0,2,6,8,12,14


### 8.3 Factor groups of $P(4)$: finding $P(4)/H \cong P(3)$

**Strategy:**
1. Find subgroups with `groups_fast.generate_subgroups_dynamic_programming`
2. Check normality by comparing left and right cosets
3. Compute factor groups for non-trivial normal subgroups
4. Test isomorphism with $P(3)$ using `groups_fast.isomorphisms_generator_backtracking`

In [ ]:
# Step 1: Find subgroups efficiently
p4_subgroups = groups_fast.generate_subgroups_dynamic_programming(
    np.array(table_p4, dtype=np.int32)
)
print(f"P(4) has {len(p4_subgroups)} subgroups")

# Step 2: Find normal subgroups (left cosets == right cosets)
sc_subgroups = []
for s in p4_subgroups:
    if groups.right_coset(table_p4, s) == groups.left_coset(table_p4, s):
        sc_subgroups.append(s)

print(f"{len(sc_subgroups)} are normal:")
for sg in sorted(sc_subgroups, key=lambda s: (len(s), min(s))):
    print(f"  {sorted(sg)}  (order {len(sg)})")

In [ ]:
# Steps 3 & 4: Find which normal subgroup gives factor group ≅ P(3)
table_p3 = groups.make_multiplication_table(groups.permutation_matrices(3))

for sg in sorted(sc_subgroups, key=lambda s: (len(s), min(s))):
    if len(sg) in (1, len(table_p4)):  # skip trivial
        continue
    _, ft = groups.factor_group(table_p4, sg)
    if len(ft) == len(table_p3):
        # Use fast isomorphism check
        found = False
        for iso in groups_fast.isomorphisms_generator_backtracking(
            np.array(ft, dtype=np.int32), np.array(table_p3, dtype=np.int32)
        ):
            found = True
            break
        print(f"H = {sorted(sg)}: P(4)/H ≅ P(3)? {found}")

In [ ]:
HTML(plot.structure_explorer(table_p4, cell_size=22))

In [ ]:
# Try a different group!


In [ ]:
# Explore subgroups and factor groups


In [ ]:
# Compare two groups for isomorphism
